Order selection intentionally gives loyalty members higher purchase frequency, weekends higher demand, and later months moderate growth. Quality issues: invalid foreign keys, inconsistent status/payment labels, and two timestamp formats.

In [ ]:
start, end = datetime(2025, 1, 1), datetime(2026, 6, 30)
dates, date_weights = [], []
cursor = start
total_days = (end - start).days + 1
while cursor <= end:
    progress = (cursor - start).days / total_days
    weight = (0.75 + progress * 0.65) * (1.32 if cursor.weekday() >= 5 else 1.0)
    if cursor.month == 12:
        weight *= 1.22
    dates.append(cursor); date_weights.append(weight); cursor += timedelta(days=1)

customer_ids = list(customer_meta)
loyalty_order_weight = {"Basic": 0.65, "Silver": 1.05, "Gold": 1.85, "Platinum": 2.70}
customer_weights = [loyalty_order_weight[customer_meta[c]["loyalty"]] for c in customer_ids]
store_ids = list(store_meta)
store_weights = [store_meta[s]["demand_weight"] for s in store_ids]

order_rows, order_meta = [], {}
for order_id in range(1, 4001):
    customer_id = rng.choices(customer_ids, weights=customer_weights, k=1)[0]
    store_id = rng.choices(store_ids, weights=store_weights, k=1)[0]
    order_day = rng.choices(dates, weights=date_weights, k=1)[0]
    order_time = order_day + timedelta(hours=rng.randint(7, 21), minutes=rng.randint(0, 59))
    status = rng.choices(["Completed", "Cancelled", "Pending"], weights=[91, 5, 4], k=1)[0]
    payment = rng.choices(["Credit Card", "Debit Card", "Digital Wallet", "Cash"], weights=[43, 26, 23, 8], k=1)[0]
    raw_customer_id = 999999 if order_id % 509 == 0 else customer_id
    raw_store_id = 9999 if order_id % 631 == 0 else store_id
    order_rows.append({
        "OrderID": order_id, "CustomerID": raw_customer_id, "StoreID": raw_store_id,
        "OrderTimestampRaw": order_time.strftime("%m/%d/%Y %H:%M") if order_id % 37 == 0 else order_time.strftime("%Y-%m-%d %H:%M:%S"),
        "OrderStatus": dirty_choice(status, [status.lower(), status.upper(), f" {status} "], order_id, 21),
        "PaymentMethod": dirty_choice(payment, [payment.lower(), payment.upper(), f" {payment} "], order_id, 33),
    })
    order_meta[order_id] = {"customer_id": customer_id, "store_id": store_id, "status": status}

orders_bronze = write_bronze(order_rows, "orders")
display(orders_bronze.limit(10))

 Normalize status and payment method.
- Parse both timestamp formats.
- Remove orders with invalid customer/store keys.
- Add calendar attributes for dashboard analysis.

In [ ]:
orders_stage = (
    bronze["orders"].dropDuplicates(["OrderID"])
    .withColumn("OrderStatus", F.initcap(F.trim(F.col("OrderStatus"))))
    .withColumn("PaymentMethod", F.initcap(F.trim(F.col("PaymentMethod"))))
    .withColumn("OrderTimestamp", F.coalesce(
        F.to_timestamp("OrderTimestampRaw", "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp("OrderTimestampRaw", "MM/dd/yyyy HH:mm"),
    ))
    .filter(F.col("OrderTimestamp").isNotNull())
)
orders_silver = (
    orders_stage
    .join(customers_silver.select("CustomerID"), "CustomerID", "inner")
    .join(stores_silver.select("StoreID"), "StoreID", "inner")
    .withColumn("OrderDate", F.to_date("OrderTimestamp"))
    .withColumn("OrderYear", F.year("OrderTimestamp"))
    .withColumn("OrderMonth", F.month("OrderTimestamp"))
    .withColumn("YearMonth", F.date_format("OrderTimestamp", "yyyy-MM"))
    .withColumn("DayName", F.date_format("OrderTimestamp", "EEEE"))
    .withColumn("IsWeekend", F.dayofweek("OrderTimestamp").isin([1, 7]))
    .select("OrderID", "CustomerID", "StoreID", "OrderTimestamp", "OrderDate", "OrderYear", "OrderMonth", "YearMonth", "DayName", "IsWeekend", "OrderStatus", "PaymentMethod")
)
write_silver(orders_silver, "orders")
display(orders_silver.groupBy("OrderStatus").count().orderBy(F.desc("count")))